# Tool Notebook · Streaming and Fine-Grained Tool Calling
This notebook demonstrates SS10 concepts: streaming tool calls, inspecting incremental `input_json` chunks, and enabling fine-grained tool calling for lower-latency argument emission.

[Open SS10 Lesson Notes: Fine Grained Tool Calling](../s06_tool_use_with_claude/ss10_fine_grained_tool_calling/index.md)
[Open SS09 Lesson Notes: Using Multiple Tools](../s06_tool_use_with_claude/ss09_using_multiple_tools/index.md)

In [9]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv(override=True)

client = Anthropic()
model = "claude-sonnet-4-5"

## 1. Initialize Client for Streaming
Loads environment variables, initializes the Anthropic client, and sets the model used for all streaming tool-use calls.

[Open SS10 Lesson Notes: Fine Grained Tool Calling](../s06_tool_use_with_claude/ss10_fine_grained_tool_calling/index.md)

In [10]:
# Helper functions


def add_user_message(messages, message):
    if isinstance(message, list):
        user_message = {
            "role": "user",
            "content": message,
        }
    else:
        user_message = {
            "role": "user",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(user_message)


def add_assistant_message(messages, message):
    if isinstance(message, list):
        assistant_message = {
            "role": "assistant",
            "content": message,
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append(
                    {
                        "type": "tool_use",
                        "id": block.id,
                        "name": block.name,
                        "input": block.input,
                    }
                )
        assistant_message = {
            "role": "assistant",
            "content": content_list,
        }
    else:
        # String messages need to be wrapped in a list with text block
        assistant_message = {
            "role": "assistant",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(assistant_message)


def chat_stream(
    messages,
    system=None,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
    betas=[],
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if betas:
        params["betas"] = betas

    return client.beta.messages.stream(**params)


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

## 2. Build Streaming Message Helpers
These helpers normalize user/assistant content blocks and provide `chat_stream`, which forwards optional `tools`, `tool_choice`, and `betas` to the streaming API.

[Open SS10 Lesson Notes: Fine Grained Tool Calling](../s06_tool_use_with_claude/ss10_fine_grained_tool_calling/index.md)

In [11]:
# Tool definition
from anthropic.types import ToolParam

save_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Eight sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)
save_short_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Review of paper. One short sentence max",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)


def save_article(**kwargs):
    return "Article saved!"


## 3. Define Tool Schemas
Defines strict tool input structures for article-saving behavior, including nested JSON fields that make streaming validation behavior easy to observe.

[Open SS10 Lesson Notes: Fine Grained Tool Calling](../s06_tool_use_with_claude/ss10_fine_grained_tool_calling/index.md)

In [12]:
# Tool Running
import json


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

## 4. Execute Requested Tools and Build Results
Routes each `tool_use` block to the local implementation and returns one `tool_result` block per request, including explicit error payloads when execution fails.

[Open SS09 Lesson Notes: Using Multiple Tools](../s06_tool_use_with_claude/ss09_using_multiple_tools/index.md)

In [13]:
# Run conversation
def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False):
    while True:
        with chat_stream(
            messages,
            tools=tools,
            betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else [],
            tool_choice=tool_choice,
        ) as stream:
            for chunk in stream:
                if chunk.type == "text":
                    print(chunk.text, end="")

                if chunk.type == "content_block_start":
                    if chunk.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

                if chunk.type == "input_json" and chunk.partial_json:
                    print(chunk.partial_json, end="")

                if chunk.type == "content_block_stop":
                    print("\n")

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        if tool_choice:
            break

    return messages

## 5. Stream Tool Calls and Fine-Grained JSON Chunks
This loop consumes streaming events, prints text/tool boundaries, and surfaces `input_json` partial chunks in real time.

When `fine_grained=True`, the beta flag `fine-grained-tool-streaming-2025-05-14` is enabled, which reduces buffering but disables API-side JSON validation.

[Open SS10 Lesson Notes: Fine Grained Tool Calling](../s06_tool_use_with_claude/ss10_fine_grained_tool_calling/index.md)

In [14]:
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

run_conversation(
    messages,
    tools=[save_article_schema],
    fine_grained=True
)

I'll create and save a fake computer science article for you.


>>> Tool Call: "save_article"
{"abstract": "This paper introduces a novel quantum-resistant cryptographic algorithm based on lattice structures that achieves O(log n) encryption time complexity.", "meta": {
  "word_count": 4750,
  "review": "This paper presents an innovative approach to post-quantum cryptography by leveraging advanced lattice-based constructions. The proposed algorithm demonstrates significant improvements in both computational efficiency and security guarantees compared to existing methods. The theoretical analysis is rigorous, providing formal proofs of security under standard lattice assumptions. Experimental results show that the implementation achieves practical performance on commodity hardware. The authors address potential side-channel vulnerabilities through careful algorithmic design. However, the paper would benefit from more extensive comparisons with recent hybrid encryption schemes. The discu

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'll create and save a fake computer science article for you."},
   {'type': 'tool_use',
    'id': 'toolu_011jzRbdM6gh9pAT3iU27KSu',
    'name': 'save_article',
    'input': {'abstract': 'This paper introduces a novel quantum-resistant cryptographic algorithm based on lattice structures that achieves O(log n) encryption time complexity.',
     'meta': {'word_count': 4750,
      'review': 'This paper presents an innovative approach to post-quantum cryptography by leveraging advanced lattice-based constructions. The proposed algorithm demonstrates significant improvements in both computational efficiency and security guarantees compared to existing methods. The theoretical analysis is rigorous, providing formal proofs of security under standard lattice assumptions. Experimental results show that the implement

## 6. Run Fine-Grained Streaming Demo
Runs an end-to-end request with `fine_grained=True` so you can observe incremental `input_json` chunks as tool arguments are generated.

If you process chunks programmatically, add defensive JSON parsing because fine-grained mode can emit temporarily invalid partial JSON.

[Open SS10 Lesson Notes: Fine Grained Tool Calling](../s06_tool_use_with_claude/ss10_fine_grained_tool_calling/index.md)